In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
%pip install librosa evaluate jiwer matplotlib pandas scikit-learn tqdm pyarrow --upgrade


In [ ]:
import torch
import psutil
import os

print("="*50)
print("       SYSTEM INFRASTRUCTURE REPORT")
print("="*50)

print("\n[Hardware Core - GPU]")
if torch.cuda.is_available():
    print(f"Entity: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Attributes: {vram:.2f} GB GDDR6 VRAM")
else:
    print("Entity: No GPU detected or CUDA not configured.")

print("\n[System Memory - RAM]")
ram = psutil.virtual_memory().total / (1024**3)
print(f"Entity: Random Access Memory (RAM)")
print(f"Attributes: {ram:.2f} GB System Allocation")

print("\n[Core Framework]")
print(f"Engine: PyTorch Framework Architecture")
print(f"Version: {torch.__version__}")

print("\n[Parallel Computing Platform]")
print(f"Ecosystem: NVIDIA CUDA Ecosystem")
print(f"Version: {torch.version.cuda if torch.cuda.is_available() else 'N/A'}")
print("="*50)


In [10]:
import torch
print("GPU Available:", torch.cuda.is_available())
print("Device Name:", torch.cuda.get_device_name(0))


GPU Available: True
Device Name: NVIDIA GeForce RTX 4090


In [11]:
import os

class Config:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()

    DATASET_DIR = os.path.join(BASE_DIR, "dataset/chunks")
    SPLIT_DIR = os.path.join(BASE_DIR, "dataset/splits")

    MODEL_DIR = os.path.join(BASE_DIR, "models")
    OUTPUT_DIR = os.path.join(BASE_DIR, "output")

    TRAIN_CSV = os.path.join(SPLIT_DIR, "train.csv")
    TEST_CSV = os.path.join(SPLIT_DIR, "test.csv")
    VAL_CSV = os.path.join(SPLIT_DIR, "val.csv")
    METADATA = os.path.join(BASE_DIR, "dataset/metadata.csv")

    SAMPLE_RATE = 16000
    BATCH_SIZE = 16
    EPOCHS = 25
    LR = 5e-5

    MODEL_NAME = "arijitx/wav2vec2-xls-r-300m-bengali"


In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

cfg = Config()

df = pd.read_csv(cfg.METADATA)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["region"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["region"]
)

# save
os.makedirs(cfg.SPLIT_DIR, exist_ok=True)

train_df.to_csv(cfg.TRAIN_CSV, index=False)
val_df.to_csv(cfg.VAL_CSV, index=False)
test_df.to_csv(cfg.TEST_CSV, index=False)

print("Stratified Split Done!")
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Stratified Split Done!
Train: 5320
Val: 665
Test: 665


In [13]:
summary = pd.DataFrame({
    "Train": train_df["region"].value_counts(),
    "Validation": val_df["region"].value_counts(),
    "Test": test_df["region"].value_counts()
}).fillna(0).astype(int)

print("\n--- Split Summary ---")
print(summary)



--- Split Summary ---
            Train  Validation  Test
region                             
barishal     1064         133   133
chittagong   1064         133   133
noakhali     1064         133   133
rangpur      1064         133   133
sylhet       1064         133   133


In [14]:
import numpy as np

def add_noise(audio, noise_factor=0.003):
    noise = np.random.randn(len(audio))
    return audio + noise_factor * noise


In [15]:
from torch.utils.data import Dataset
import librosa

class Wav2VecDataset(Dataset):
    def __init__(self, csv_file, config, processor, augment=False):
        self.df = pd.read_csv(csv_file)
        self.cfg = config
        self.processor = processor
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def load_audio(self, path):
        audio, sr = librosa.load(path, sr=self.cfg.SAMPLE_RATE)
        audio = np.nan_to_num(audio)
        audio = audio[:self.cfg.SAMPLE_RATE * 10]  # LIMIT LENGTH TO 10 SECONDS
        return audio

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.cfg.BASE_DIR, row["audio_path"])

        if not os.path.exists(path):
            raise FileNotFoundError(path)

        audio = self.load_audio(path)

        if self.augment:
            if np.random.rand() < 0.3:
                audio = add_noise(audio)

        inputs = self.processor(
            audio,
            sampling_rate=self.cfg.SAMPLE_RATE,
            return_tensors="pt",
            padding=True
        )
        input_values = inputs.input_values[0]

        labels = self.processor.tokenizer(
            row["transcript"],
            return_tensors="pt"
        ).input_ids.squeeze()

        return {
            "input_values": input_values,
            "labels": labels,
            "region": row["region"]
        }


In [17]:
def collate_fn(batch):
    input_values = torch.nn.utils.rnn.pad_sequence(
        [b["input_values"] for b in batch],
        batch_first=True
    )
    labels = torch.nn.utils.rnn.pad_sequence(
        [b["labels"] for b in batch],
        batch_first=True,
        padding_value=-100
    )
    regions = [b["region"] for b in batch]

    return {
        "input_values": input_values,
        "labels": labels,
        "region": regions
    }


In [19]:
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = Wav2Vec2Processor.from_pretrained(cfg.MODEL_NAME)
model = Wav2Vec2ForCTC.from_pretrained(cfg.MODEL_NAME)

model.to(device)
print("Device:", device)

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Device: cuda


In [20]:
from torch.utils.data import DataLoader

train_ds = Wav2VecDataset(cfg.TRAIN_CSV, cfg, processor, augment=True)
val_ds = Wav2VecDataset(cfg.VAL_CSV, cfg, processor, augment=False)
test_ds = Wav2VecDataset(cfg.TEST_CSV, cfg, processor, augment=False)

train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, num_workers=0, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=cfg.BATCH_SIZE, num_workers=0, collate_fn=collate_fn)


In [21]:

df_check = pd.read_csv(cfg.TRAIN_CSV)
missing = [os.path.join(cfg.BASE_DIR, p) for p in df_check["audio_path"] if not os.path.exists(os.path.join(cfg.BASE_DIR, p))]
print("Missing Files:", len(missing))
if len(missing) > 0:
    print(missing[:10])


Missing Files: 0


In [11]:
import torch.optim as optim
from tqdm import tqdm
from transformers import get_linear_schedule_with_warmup 

train_losses = []
val_losses = []

optimizer = optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=0.01)

num_training_steps = len(train_loader) * cfg.EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=500,
    num_training_steps=num_training_steps
)

best_loss = float("inf")
patience = 5  
patience_counter = 0

os.makedirs(cfg.MODEL_DIR, exist_ok=True)

for epoch in range(cfg.EPOCHS):
    print("🚀 Epoch:", epoch + 1)

    # এনকোডার ফ্রিজিং এবং আনফ্রিজিং লজিক (Wav2Vec2 এর জন্য)
    if epoch < 3:
        print("Feature Extractor/Encoder Frozen for stabilization.")
        for param in model.wav2vec2.feature_extractor.parameters():
            param.requires_grad = False
    else:
        print("Feature Extractor/Encoder Unfrozen.")
        for param in model.wav2vec2.feature_extractor.parameters():
            param.requires_grad = True

    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader)

    for step, batch in enumerate(progress_bar):
        batch = {k: v.to(device) for k, v in batch.items() if k != "region"}

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # গ্রেডিয়েন্ট ক্লিপিং

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        progress_bar.set_postfix({"loss": loss.item()})

    avg_train_loss = total_loss / len(train_loader)
    print("Train Loss:", avg_train_loss)

    # VALIDATION
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for batch in tqdm(val_loader):
            batch = {k: v.to(device) for k, v in batch.items() if k != "region"}
            outputs = model(**batch)
            val_loss += outputs.loss.item()

    avg_val_loss = val_loss / len(val_loader)
    print("Val Loss:", avg_val_loss)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    
    # আর্লি স্টপিং ও বেস্ট চেকপয়েন্ট সেভিং
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        patience_counter = 0
        checkpoint_path = os.path.join(cfg.MODEL_DIR, "best_wav2vec_model.pt")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': best_loss,
        }, checkpoint_path)
        print(f"Best model saved with Val Loss: {best_loss:.4f}")
    else:
        patience_counter += 1
        print(f"Loss didn't improve. Early stopping counter: {patience_counter}/{patience}")

    if patience_counter >= patience:
        print("Early stopping triggered. Training terminated!")
        break
    
    # সর্বশেষ মডেল সেভ
    torch.save(model.state_dict(), os.path.join(cfg.MODEL_DIR, "wav2vec_last.pth"))


🚀 Epoch: 1
Feature Extractor/Encoder Frozen for stabilization.


  0%|          | 0/333 [00:00<?, ?it/s]c:\Users\USER\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,
100%|██████████| 333/333 [31:37<00:00,  5.70s/it, loss=2.36]  


Train Loss: 2.914906443776311


100%|██████████| 42/42 [02:57<00:00,  4.23s/it]


Val Loss: 2.0063812051500594
Best model saved with Val Loss: 2.0064
🚀 Epoch: 2
Feature Extractor/Encoder Frozen for stabilization.


...

Val Loss: 1.3202996651331584
Best model saved with Val Loss: 1.3203
🚀 Epoch: 25
Feature Extractor/Encoder Unfrozen.


100%|██████████| 333/333 [16:05<00:00,  2.90s/it, loss=1.24]


Train Loss: 1.7496535276865457


100%|██████████| 42/42 [02:45<00:00,  3.93s/it]


Val Loss: 1.3201321391832261
Best model saved with Val Loss: 1.3201


In [6]:
import matplotlib.pyplot as plt

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(8,5))
plt.plot(epochs, train_losses, label="Train Loss")
plt.plot(epochs, val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Wav2Vec2 Training vs Validation Loss Curve")
plt.legend()
plt.grid()

save_path = os.path.join(cfg.OUTPUT_DIR, "loss_curve.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight') 
print(f"Loss curve graph successfully saved at: {save_path}")
plt.show()


NameError: name 'train_losses' is not defined

In [22]:
import evaluate
import re
from collections import defaultdict

checkpoint_path = os.path.join(cfg.MODEL_DIR, "best_wav2vec_model.pt")
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print("Loaded best weights for evaluation.")

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def clean_bengali_text(text):
    text = re.sub(r'[।,;:!?•\'"()\[\]{}—\-_]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


Loaded best weights for evaluation.


In [23]:
def evaluate_model(loader):
    model.eval()
    preds = []
    refs = []

    with torch.no_grad():
        for batch in loader:
            inputs = batch["input_values"].to(device)

            logits = model(inputs).logits
            predicted_ids = torch.argmax(logits, dim=-1)

            pred_text = processor.batch_decode(predicted_ids)

            labels = batch["labels"].clone()
            labels[labels == -100] = processor.tokenizer.pad_token_id
            ref_text = processor.batch_decode(labels)

            cleaned_pred_text = [clean_bengali_text(t) for t in pred_text]
            cleaned_ref_text = [clean_bengali_text(t) for t in ref_text]

            preds.extend(cleaned_pred_text)
            refs.extend(cleaned_ref_text)
            
    wer = wer_metric.compute(predictions=preds, references=refs)
    cer = cer_metric.compute(predictions=preds, references=refs)
    
    word_accuracy = max(0, (1 - wer) * 100)
    char_accuracy = max(0, (1 - cer) * 100)

    return wer, cer, word_accuracy, char_accuracy


In [24]:
wer, cer, w_acc, c_acc = evaluate_model(test_loader)

print("FINAL RESULT")
print("FINAL TEST DATA RESULT")
print(f"Word Error Rate (WER)     : {wer:.4f}")
print(f"Character Error Rate (CER): {cer:.4f}")
print(f"Word Accuracy (WAcc)      : {w_acc:.2f}%") 
print(f"Character Accuracy (CAcc) : {c_acc:.2f}%") 

txt_save_path = os.path.join(cfg.OUTPUT_DIR, "evaluation_results.txt")
with open(txt_save_path, "w", encoding="utf-8") as f:
    f.write("FINAL RESULT\n")
    f.write("FINAL TEST DATA RESULT\n")
    f.write(f"Word Error Rate (WER)     : {wer:.4f}\n")
    f.write(f"Character Error Rate (CER): {cer:.4f}\n")
    f.write(f"Word Accuracy (WAcc)      : {w_acc:.2f}%\n")
    f.write(f"Character Accuracy (CAcc) : {c_acc:.2f}%\n")

print(f"\nEvaluation results successfully saved at: {txt_save_path}")


c:\Users\USER\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


FINAL RESULT
FINAL TEST DATA RESULT
Word Error Rate (WER)     : 0.5505
Character Error Rate (CER): 0.3150
Word Accuracy (WAcc)      : 44.95%
Character Accuracy (CAcc) : 68.50%

Evaluation results successfully saved at: d:\TEAM_34\output\evaluation_results.txt


In [25]:
from jiwer import wer as jiwer_wer

region_refs = defaultdict(list)
region_preds = defaultdict(list)

model.eval()
with torch.no_grad():
    for batch in test_loader: 
        region = batch["region"] 
        inputs = batch["input_values"].to(device)

        logits = model(inputs).logits
        predicted_ids = torch.argmax(logits, dim=-1)
        preds = processor.batch_decode(predicted_ids)

        labels = batch["labels"].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id
        refs = processor.batch_decode(labels)

        for r, pred, ref in zip(region, preds, refs):
            region_preds[r].append(clean_bengali_text(pred))
            region_refs[r].append(clean_bengali_text(ref))

regional_txt_path = os.path.join(cfg.OUTPUT_DIR, "regional_accuracy_report.txt")
print("\nREGIONAL ACCURACY REPORT")

with open(regional_txt_path, "w", encoding="utf-8") as f:
    f.write("REGIONAL ACCURACY REPORT\n")
    f.write("=========================\n\n")
    
    for region in region_refs:
        region_wer = jiwer_wer(region_refs[region], region_preds[region])
        region_w_acc = max(0, (1 - region_wer) * 100) 
        
        print(f"{region}:")
        print(f"   - WER      : {region_wer:.4f}")
        print(f"   - Accuracy : {region_w_acc:.2f}%")
        
        f.write(f"{region}:\n")
        f.write(f"   - WER      : {region_wer:.4f}\n")
        f.write(f"   - Accuracy : {region_w_acc:.2f}%\n\n")

print(f"\nRegional accuracy report successfully saved at: {regional_txt_path}")


REGIONAL ACCURACY REPORT
chittagong:
   - WER      : 0.5279
   - Accuracy : 47.21%
sylhet:
   - WER      : 0.4899
   - Accuracy : 51.01%
barishal:
   - WER      : 0.4177
   - Accuracy : 58.23%
rangpur:
   - WER      : 0.5897
   - Accuracy : 41.03%
noakhali:
   - WER      : 0.7308
   - Accuracy : 26.92%

Regional accuracy report successfully saved at: d:\TEAM_34\output\regional_accuracy_report.txt
